# A single-compartment neuron with Hodgkin & Huxley and transient K+ conductances

The squid axon gets by with two active conductances. Real neurons have
dozens, and one of the most consequential is $I_A$ -- a K+ current that
**inactivates**, like the sodium current does, and so is only available
after the cell has been held hyperpolarised.

We first clamp $I_A$ to see what it is, then put it back into a firing cell
to see what it does.

This notebook grades your answers for you, and **each student gets a
slightly different neuron** -- including your own recording temperature.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON
#!pip install neuron quantities

In [ ]:
# Fetch mechanisms
# Uncomment this line if on google colab
#!git clone https://github.com/ABL-Lab/NSC6084-A26.git

In [ ]:
# Compile the mechanisms
# Note: recompiled mechanisms will not take effect until neuron is imported or the jupyter kernel is restarted

# Uncomment this line if on google colab
#!nrnivmodl ./NSC6084-A26/Sept15/mechanisms
# Uncomment this line if running locally
!nrnivmodl mechanisms

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s, MOhm

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

## Step 1b: Load your personal exercise parameters

Kinetics are strongly temperature dependent, and **your cell is recorded at
your own temperature** -- which is what makes the time constant you measure
in Question 1 yours and not your neighbour's.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

soma_length = assignment.params["soma_length_um"]
g_leak      = assignment.params["g_leak_nS"]
ia_celsius  = assignment.params["ia_celsius"]     # your recording temperature
ia_step_v   = assignment.params["ia_step_v_mV"]   # Question 1 steps here
ia_hold_v   = assignment.params["ia_hold_v_mV"]   # Question 2 holds here

print(f"Your soma length:           {soma_length} um")
print(f"Your leak conductance:      {g_leak} nS")
print(f"Your temperature:           {ia_celsius} degC")
print(f"Your Question 1 step:       {ia_step_v} mV")
print(f"Your Question 2 holding V:  {ia_hold_v} mV")
print(f"Exercises to submit:        {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# soma.L is YOUR personal value, loaded in Step 1b above.
soma.L = soma_length * um
soma.diam =  10 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

In [ ]:
area

In [ ]:
volume

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)
    sec.Ra = 100

### Add the Hodgkin-Huxley conductances

In [ ]:
# This model includes the transient Na+, persistent K+ and the leak conductances
soma.insert("hh")

That's almost too easy!

### Parametize the leak conductance G = 1/R

In [ ]:
G = g_leak * nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
v_rest = -70*mV

In [ ]:
tau_m = (specific_membrane_capacitance * area / G).rescale(ms)

In [ ]:
tau_m = soma(0.5).cm / soma(0.5).hh.gl

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.hh.gl = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.hh.el = -54.3

In [ ]:
# Read it back off the model, now that gl has actually been assigned, and
# check it against the tau_m we computed from G above.
tau_m = ((soma(0.5).cm * uF/cm**2) / (soma(0.5).hh.gl * S/cm**2)).rescale(ms)
tau_m

### Inspect our parameters

In [ ]:
soma.psection()

In [ ]:
soma.nseg

### Add a current injection

In [ ]:
stim = h.IClamp(soma(0.5))

In [ ]:
stim.delay = 200 * ms  # Inject current 500ms after the start of the simulation 
stim.dur = 600 * ms  # stop injecting current at 520ms 
stim.amp = 0.032 * nA  # Inject 0.1 nA of current

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

In [ ]:
# Record hh gating variables
hh_vars = ['h', 'm', 'n', 'gna', 'gk']
hh_recordings = {}
for var in hh_vars:
    ref = getattr(soma(0.5).hh, "_ref_"+var )
    hh_recordings[var] = h.Vector().record(ref) 

In [ ]:
hh_recordings

### Run the simulation

In [ ]:
h.finitialize( float(v_rest) )
h.continuerun( float(1000 * ms) )

## Step 4: Plot the results

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.axis([0,1000,-80,30])

In [ ]:
plt.plot(t, hh_recordings['m'], lw=2, label="m")
plt.plot(t, hh_recordings['h'], lw=2, label="h")
plt.plot(t, hh_recordings['n'], lw=2, label="n")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("fraction", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.axis([195,220,0,1])

In [ ]:
plt.plot(t, hh_recordings['gna'], lw=2, label="gna")
plt.plot(t, hh_recordings['gk'], lw=2, label="gk")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("fraction", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.axis([195,220,0, 0.05])

In [ ]:
def find_spikes(v, t):
    """ Returns times of spikes for a voltage trace and time grid"""
    
    # look for upward crossing of 0mV
    v_arr = np.array(v)
    t_arr = np.array(t) 
    # This is tricky & powerful notation! Let's discuss in class!
    return t_arr[1:][(v_arr[1:]>0) & (v_arr[:-1]<0)] 

In [ ]:
spike_times = find_spikes(soma_v, t)

In [ ]:
spike_times, len(spike_times)

In [ ]:
firing_freq = (len(spike_times)/(stim.dur*ms)).rescale(Hz)

In [ ]:
firing_freq

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.plot(spike_times, len(spike_times)*[0], 'r.')
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.axis([200,400,-80,30])

### The f-I curve without $I_A$

Sweep the injected current and measure the firing rate. Keep this curve --
we will put $I_A$ in and draw it again at the end of the notebook.

In [ ]:
I_range = np.arange(0,0.1,0.001)

In [ ]:
def find_freq(I):
    stim.amp = I
    h.finitialize( float(v_rest) )
    h.continuerun( float(1000 * ms) )
    spike_times = find_spikes(soma_v, t)
    firing_freq = (len(spike_times[spike_times>200])/(stim.dur*ms)).rescale(Hz)
    return firing_freq

In [ ]:
# Note this cool notation: List comprehension
freqs_no_ia = [find_freq(x) for x in I_range]

In [ ]:
plt.plot(I_range, freqs_no_ia, 'x')
plt.xlabel("injected current [nA]", size=14)
plt.ylabel("firing rate [Hz]", size=14)
plt.title("no $I_A$", size=13)

---

## Part 2 -- What is $I_A$? Clamp it and find out

Before we let $I_A$ near a firing cell, let us look at the current on its own
under voltage clamp -- the same way we pulled the Na+ and K+ currents apart in
`HH_neuron2_voltage_clamp.ipynb`.

We will use two different transient K+ channel models:

- **`K_Tst`**, the transient K+ component as in the Connor & Stevens model
  (see Dayan & Abbott, pg 196);
- **`Kv4_2_0016`**, a model of the genetically identified channel **Kv4.2**
  from channelpedia -- a real, cloned channel rather than a curve fit.

The questions use the Kv4.2 model.

In [ ]:
# Switch the current injection off -- the clamp takes over from here.
stim.amp = 0 * nA

vclamp = h.SEClamp(soma(0.5))
T_STEP = 200 * ms        # the step begins here
vclamp.dur1 = T_STEP     # hold at amp1 until then
vclamp.dur2 = 700 * ms   # hold the step voltage for 700 ms
vclamp.amp1 = v_rest     # holding potential before the step
vclamp.amp2 = -40*mV     # the step voltage (overwritten by the runs below)
vclamp.amp3 = v_rest     # back to rest afterwards
vclamp.rs = 0.01 * MOhm  # series resistance, < 1/100 of Rin

vclamp_i = h.Vector().record(vclamp._ref_i)

In [ ]:
def set_hh(gnabar=0.12, gkbar=0.036):
    """ Switch the HH Na+ and K+ conductances on or off (units: S/cm2). """
    for seg in soma:
        seg.hh.gnabar = gnabar
        seg.hh.gkbar = gkbar

def run_clamp(step_voltage, holding_voltage=None):
    """ Step to step_voltage at T_STEP; return (t, v, i_clamp) as arrays. """
    if holding_voltage is not None:
        vclamp.amp1 = holding_voltage
    vclamp.amp2 = step_voltage
    h.finitialize( float(vclamp.amp1) )
    h.continuerun( float(1000 * ms) )
    return np.array(t), np.array(soma_v), np.array(vclamp_i)

### The Connor & Stevens transient K+ current

In [ ]:
soma.insert("K_Tst")
soma(0.5).K_Tst.gK_Tstbar = 0.1*0.477

In [ ]:
set_hh(gnabar=0.0, gkbar=0.0)  # only the transient K+ current and the leak
fig = plt.figure()
ax1, ax2 = fig.subplots(2, 1)
for step_voltage in [-70, -60, -50, -40, -30, -20, -10, 0]:
    a_t, v, i = run_clamp(step_voltage, holding_voltage=v_rest)
    ax1.plot(a_t, i, lw=2, label="%f mV" % step_voltage)
    ax2.plot(a_t, v, lw=2, label="%f mV" % step_voltage)
ax1.set_xlabel("t [ms]", size=16)
ax1.set_ylabel("i [nA]", size=16)
ax2.set_xlabel("t [ms]", size=16)
ax2.set_ylabel("v [mV]", size=16)
ax1.axis([195,225,-0.1,0.6])
ax2.axis([195,225, -80, 20])

Outward, like the delayed rectifier -- but unlike the delayed rectifier (HH K+ channel), it **decays away while the voltage is still held**. That is inactivation, and it is a key property of "A-type" K+ channels.

### The Kv4.2 channel from channelpedia

In [ ]:
soma.insert("Kv4_2_0016")
soma(0.5).Kv4_2_0016.gKv4_2bar = 0.1*0.8
soma(0.5).Kv4_2_0016.q10 = 3.0

# Zero the other transient K+ channel, so we are looking at Kv4.2 alone
soma(0.5).K_Tst.gK_Tstbar = 0

# Your own recording temperature, loaded in Step 1b
h.celsius = ia_celsius
h.celsius

In [ ]:
set_hh(gnabar=0.0, gkbar=0.0)  # Kv4.2 and the leak only
fig = plt.figure()
ax1, ax2 = fig.subplots(2, 1)
for step_voltage in [-70, -60, -50, -40, -30, -20, -10, 0]:
    a_t, v, i = run_clamp(step_voltage, holding_voltage=v_rest)
    ax1.plot(a_t, i, lw=2, label="%f mV" % step_voltage)
    ax2.plot(a_t, v, lw=2, label="%f mV" % step_voltage)
ax1.set_xlabel("t [ms]", size=16)
ax1.set_ylabel("i [nA]", size=16)
ax2.set_xlabel("t [ms]", size=16)
ax2.set_ylabel("v [mV]", size=16)
ax1.axis([195,260,-0.1,1.2])
ax2.axis([195,260, -80, 20])

### Temperature changes the kinetics -- and *only* the kinetics

`Kv4_2_0016` has a `q10` of 3: every time constant is divided by
$q_{10}^{(T-23)/10}$, so the current runs about three times faster for every
10 degrees. The steady-state curves $m_\infty(V)$ and $h_\infty(V)$ have no
temperature term at all.

Watch what that does and does not change in the plot below.

In [ ]:
set_hh(gnabar=0.0, gkbar=0.0)
plt.figure()
for temp in [15, 25, 35]:
    h.celsius = temp
    a_t, v, i = run_clamp(0, holding_voltage=v_rest)
    plt.plot(a_t, i, lw=1.5, label="%.1f degC" % temp)
h.celsius = ia_celsius   # put your own temperature back
plt.legend()
plt.xlabel("t [ms]", size=14)
plt.ylabel("i [nA]", size=14)
plt.axis([195, 260, -0.1, 1.2])

The three currents rise and fall at completely different speeds -- and reach
**the same peak**. Cooling the cell does not give you more current, it gives
you the same current spread over more time.

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** neuron, using the parameters printed in
Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM;
you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, fading to
zero at 20% off.

### Question 1 -- How fast does $I_A$ inactivate?

Step to **your** `ia_step_v` at **your** temperature, and measure the time
constant with which the Kv4.2 current decays from its peak. Report it in
**ms**.

The recipe is the standard one for an exponential decay: subtract the steady
baseline, take the logarithm of what is left, and fit a straight line. The
slope is $-1/\tau$.

> Start the fit a little after the peak. Right at the peak the activation
> gate $m$ is still settling, and the decay is not a single exponential until
> it has.

In [ ]:
SKIP_MS = 0.2   # step past the capacitive artifact at the step edge

def ia_current(step_voltage, holding_voltage=None):
    """ Leak-subtracted Kv4.2 current after a step, as (time_since_step, i). """
    a_t, v, i = run_clamp(step_voltage, holding_voltage)
    t0 = float(T_STEP)
    late = i[np.argmin(np.abs(a_t - (t0 + 400)))]   # fully inactivated: leak only
    window = (a_t >= t0 + SKIP_MS) & (a_t <= t0 + 400)
    return a_t[window] - t0, i[window] - late

def inactivation_tau(step_voltage, holding_voltage=None):
    """ Single-exponential time constant of the decay from the peak, in ms. """
    dt, di = ia_current(step_voltage, holding_voltage)
    k = np.argmax(di)
    t_peak, i_peak = dt[k], di[k]
    # Fit from 2 ms past the peak, down to 2% of it -- below that we are
    # fitting the numerical dust on top of the leak.
    fit = (dt >= t_peak + 2.0) & (dt <= t_peak + 40.0) & (di > 0.02 * i_peak)
    slope, _ = np.polyfit(dt[fit] - t_peak, np.log(di[fit]), 1)
    return -1.0 / slope

In [ ]:
set_hh(gnabar=0.0, gkbar=0.0)
h.celsius = ia_celsius

tau_h = None # fill in

print(f"At {ia_celsius} degC, I_A inactivates with tau = {tau_h:.3f} ms")
assignment.submit(tau_h, "ia_tau_h")

Sanity check worth doing: run `inactivation_tau` again with `h.celsius` set to
23 degrees, and divide. You should get $3^{(T-23)/10}$ for your own $T$ --
the $q_{10}$, measured rather than looked up.

### Question 2 -- How much $I_A$ is available?

$I_A$ inactivates at rest. Hold the cell at a given potential long enough for
the inactivation gate to settle, then step to a fixed test voltage: the peak
current you get is proportional to how much $I_A$ was **available** at that
holding potential.

Hold at **your** `ia_hold_v`, and report the peak current as a **fraction** of
the peak you get from a deeply hyperpolarised hold of $-120$ mV, where
essentially all the channels are de-inactivated. It is a ratio, so no units.

This is the number that explains what $I_A$ is *for*. At rest there is barely
any of it; release the cell from inhibition and suddenly there is a lot -- so
$I_A$ opposes exactly the first spike after a hyperpolarisation, and delays it.

In [ ]:
TEST_V = 0.0      # step to the same test voltage every time
REFERENCE_HOLD = -120.0   # deeply hyperpolarised: (almost) nothing inactivated

def available_ia(holding_voltage):
    """ Peak Kv4.2 current after a step to TEST_V from this holding potential. """
    dt, di = ia_current(TEST_V, holding_voltage)
    return di.max()

set_hh(gnabar=0.0, gkbar=0.0)
h.celsius = ia_celsius

availability = None # fill in

print(f"Holding at {ia_hold_v} mV leaves {100*availability:.1f}% of I_A available")
assignment.submit(availability, "ia_availability")

Plot the whole curve while you are here -- hold at a range of potentials and
you have measured the **steady-state inactivation curve** of Kv4.2, the same
kind of curve you measured for the sodium current in the voltage-clamp
notebook.

In [ ]:
holds = np.arange(-120, -49, 5.0)
ref = available_ia(REFERENCE_HOLD)
plt.plot(holds, [available_ia(vh)/ref for vh in holds], 'o-')
plt.axvline(ia_hold_v, color='C1', ls='--', label='your holding potential')
plt.xlabel("holding potential [mV]", size=14)
plt.ylabel("fraction of $I_A$ available", size=14)
plt.legend()

---

## Part 3 -- What $I_A$ does to firing

Now put the cell back together: HH conductances on, clamp off, current
injection back, and Kv4.2 still there. Then redraw the f-I curve and compare
it with the one from Part 1.

In [ ]:
# Disable the voltage clamp by giving it no duration, and restore the cell.
vclamp.dur1 = vclamp.dur2 = vclamp.dur3 = 0
set_hh(gnabar=0.12, gkbar=0.036)
soma(0.5).Kv4_2_0016.gKv4_2bar = 0.1*0.8
h.celsius = ia_celsius
soma.psection()

In [ ]:
freqs_with_ia = [find_freq(x) for x in I_range]

In [ ]:
plt.plot(I_range, freqs_no_ia, 'x', label="no $I_A$")
plt.plot(I_range, freqs_with_ia, '+', label="with $I_A$")
plt.xlabel("injected current [nA]", size=14)
plt.ylabel("firing rate [Hz]", size=14)
plt.legend()

### What changed?

Two things to look for, and they are the reason $I_A$ matters:

1. The curve has moved **right** -- $I_A$ is an outward current, so it takes
   more injected current to reach threshold.
2. More interestingly, look at the **bottom** of the curve. Without $I_A$ the
   cell switches on abruptly at tens of Hz and cannot fire slowly at all
   (the type II behaviour from `Sept10/HH_neuron.ipynb`). With enough $I_A$
   the onset becomes gradual, and arbitrarily low firing rates become
   possible -- **type I** behaviour, which is what most cortical neurons do.

Connor and Stevens introduced $I_A$ for exactly this reason: the
Hodgkin-Huxley model could not reproduce the slow, smoothly graded firing
they were recording, and a transient K+ current fixed it.